# Suite RCH — Reachable

`Reachable.of(root)` returns the root and every object reachable through adjacencies, in first-reference order. It is
schema-agnostic, so most cases use hand-written `Fake` visitables that no schema describes.

In [ ]:
import sys

from schemas.Framework import Proxies, Reachable, Schemas as S
from support import Fake, raises, text


def names(objs):
    return [o.values["n"] for o in objs]


def node(n, **adjacencies):
    return Fake("Anything", {"n": n}, adjacencies)


def link(obj, *targets, name="out", **props):
    obj.adjacencies.setdefault(name, []).extend({"to": t, **props} for t in targets)

## RCH-01 · A lone object reaches only itself

In [ ]:
solo = node("solo")
assert Reachable.of(solo) == [solo]

## RCH-02 · Breadth-first, first-reference order

In [ ]:
a, b, c, d, e = (node(n) for n in "abcde")
link(a, b, c)
link(b, d)
link(c, e)
assert names(Reachable.of(a)) == ["a", "b", "c", "d", "e"]
assert names(Reachable.of(b)) == ["b", "d"]  # adjacency direction as written by accept()

## RCH-03 · Cycles, self-loops and diamonds terminate and list each object once

In [ ]:
p, q, r = node("p"), node("q"), node("r")
link(p, q)
link(q, r)
link(r, p)
link(p, p)
assert names(Reachable.of(p)) == ["p", "q", "r"]
top, left, right, bottom = (node(n) for n in ["top", "left", "right", "bottom"])
link(top, left, right)
link(left, bottom)
link(right, bottom)
assert names(Reachable.of(top)) == ["top", "left", "right", "bottom"]

## RCH-04 · Several adjacencies, several links per entry, and properties are ignored

In [ ]:
hub, x, y, z = (node(n) for n in ["hub", "x", "y", "z"])
hub.adjacencies["trio"] = [{"first": x, "second": y, "weight": 1.5, "tag": "t"}]
hub.adjacencies["other"] = [{"to": z, "blob": b"\x00"}]
assert names(Reachable.of(hub)) == ["hub", "x", "y", "z"]

## RCH-05 · Identity decides sameness, not Python object identity or equality

In [ ]:
twin1, twin2 = Fake("T", {"n": "t1"}, identity="same"), Fake("T", {"n": "t2"}, identity="same")
root = node("root")
link(root, twin1, twin2)
assert names(Reachable.of(root)) == ["root", "t1"]  # the second object with the same identity is not revisited
eq1, eq2 = node("eq"), node("eq")
link(root, eq1, eq2)
assert len(Reachable.of(root)) == 4  # equal content but distinct identities are distinct objects

## RCH-06 · The root is always first, even when others link back to it

In [ ]:
back = node("back")
start = node("start")
link(start, back)
link(back, start)
assert Reachable.of(start)[0] is start and Reachable.of(back)[0] is back

## RCH-07 · Long chains do not recurse

In [ ]:
chain = [node(str(i)) for i in range(20_000)]
for left_, right_ in zip(chain, chain[1:]):
    link(left_, right_)
assert len(Reachable.of(chain[0])) == 20_000
assert sys.getrecursionlimit() < 20_000

## RCH-08 · Errors raised by `accept` propagate

In [ ]:
class Broken(Fake):
    def accept(self, visitor):
        raise RuntimeError("broken accept")


with raises(RuntimeError, match="broken accept"):
    Reachable.of(Broken("B"))
root2 = node("root2")
link(root2, Broken("B"))
with raises(RuntimeError):
    Reachable.of(root2)

## RCH-09 · The collector refuses operations it cannot answer

In [ ]:
collector = Reachable._Collector(lambda target: None)
with raises(NotImplementedError):
    collector.adjacency("x", lambda a: a.me())
with raises(NotImplementedError):
    collector.adjacency("x", lambda a: a.add(lambda e: e.link("l", lambda k: k.target(print))))
with raises(AttributeError):
    collector.property("x", lambda p: p.value(lambda v: v.as_native(lambda n: n.get())))

## RCH-10 · Proxies: the same component from every member

In [ ]:
Rel = S.OfRelation.Builder().links("a", "b").create()
Node = S.OfObject.Builder().properties(text("n")).relations(lambda r: r.name("out").of(Rel).me("a"),
                                                            lambda r: r.name("in").of(Rel).me("b")).create()
Proxies.register("Node", Node)
Proxies.register("Rel", Rel)
B = Proxies.Builders
n1 = B.Node().n("1").create()
n2 = B.Node().n("2").out(lambda x: x.b(n1)).create()
n3 = B.Node().n("3").out(lambda x: x.b(n2)).create()
loner = B.Node().n("loner").create()
component = {o.identity() for o in Reachable.of(n1)}
assert component == {o.identity() for o in Reachable.of(n2)} == {o.identity() for o in Reachable.of(n3)}
assert len(component) == 3 and Reachable.of(loner) == [loner]